# Polarization Camera – θ/φ Reconstruction

Pipeline: **load → demosaic → channel correction → anisotropy → centering → φ, θ**

Three data sources supported:
1. **Recorder raw** — `.npy` chunks (pixels zeroed outside circle)
2. **Cycler raw** — `_raw.npz` per dwell (full rectangle, no masking)
3. **Cycler means** — `.npz` per dwell (pre-computed c0/c45/c90/c135)

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 3)})

## Configuration

In [ ]:
# ── Manual parameters ─────────────────────────────────────────────────────────
# Background offsets per channel (set manually or from dark frame).
b_0, b_45, b_90, b_135 = 0.0, 0.0, 0.0, 0.0

# Anisotropy center correction (manual only for now).
center_ax, center_ay = 0.0, 0.0

# Objective / medium (default NA — no calibration step).
nw     = 1.33     # refractive index of immersion medium
NA_obj = 1.3      # objective NA

# ── Mosaic layout (sensor row%2, col%2) ───────────────────────────────────────
# (0,0)=90°  (0,1)=45°  (1,0)=135°  (1,1)=0°
MOSAIC = {(0, 0): "90", (0, 1): "45", (1, 0): "135", (1, 1): "0"}

## Loader 1 — Recorder raw pixels (`.npy` chunks)

In [ ]:
def load_recorder_raw(meta_json: Path):
    """Load recorder output → (c0, c45, c90, c135, t) arrays of per-frame channel means."""
    with open(meta_json) as f:
        meta = json.load(f)

    prefix = str(meta_json).replace("_meta.json", "")

    # Load and concatenate chunks
    chunks = sorted(Path(meta_json).parent.glob(Path(prefix).name + "_chunk*.npy"))
    frames = np.concatenate([np.load(c) for c in chunks], axis=0)  # (N, H, W) uint16

    # Timestamps
    ts_path = Path(prefix + "_timestamps.npy")
    t = np.load(ts_path) if ts_path.exists() else np.arange(len(frames), dtype=np.float64)
    t = t[:len(frames)] - t[0]  # relative

    # Sensor origin of the crop
    sy0, sx0 = meta["crop_top_left_sensor_yx"]

    return _demosaic_frames(frames, sy0, sx0), t


# ── Example ───────────────────────────────────────────────────────────────────
# c0, c45, c90, c135, t = load_recorder_raw(Path("captures/spot1_20260309_122513/spot_20260309_122513_meta.json"))
# print(f"Loaded {len(t)} frames, duration {t[-1]:.2f} s")

## Loader 2 — Cycler raw pixels (`_raw.npz`)

In [ ]:
def load_cycler_raw(npz_path: Path):
    """Load cycler raw-pixel dwell → (c0, c45, c90, c135, t) arrays of per-frame channel means."""
    # Companion JSON for sensor origin
    json_path = npz_path.with_suffix(".json")
    with open(json_path) as f:
        meta = json.load(f)

    data = np.load(npz_path)
    frames = data["frames"]  # (N, H, W) uint16
    t = data["t"]            # relative timestamps

    sy0, sx0 = meta["crop_top_left_sensor_yx"]

    return _demosaic_frames(frames, sy0, sx0), t


# ── Example ───────────────────────────────────────────────────────────────────
# c0, c45, c90, c135, t = load_cycler_raw(Path("cycles/cycle_20260309_122513/cycle_spot01/cycle_s01_d0001_raw.npz"))
# print(f"Loaded {len(t)} frames")

## Loader 3 — Cycler means (`.npz`)

In [ ]:
def load_cycler_means(npz_path: Path):
    """Load cycler pre-computed channel means → (c0, c45, c90, c135, t)."""
    data = np.load(npz_path)
    t = data["t"]
    c0   = data["c0"]
    c45  = data["c45"]
    c90  = data["c90"]
    c135 = data["c135"]
    return (c0, c45, c90, c135), t


# ── Example ───────────────────────────────────────────────────────────────────
# c0, c45, c90, c135, t = load_cycler_means(Path("cycles/cycle_20260309_122513/cycle_spot01/cycle_s01_d0001_p0000.npz"))
# print(f"Loaded {len(t)} frames")

## Shared demosaic helper

In [ ]:
def _demosaic_frames(frames, sy0: int, sx0: int):
    """Extract per-channel means from raw mosaic frames.

    frames : (N, H, W) uint16 — raw mosaic pixels
    sy0, sx0 : absolute sensor row/col of crop top-left corner

    Returns (c0, c45, c90, c135) — each shape (N,) float64.
    """
    N, H, W = frames.shape
    # Build mosaic channel masks based on sensor coordinates
    row_mod = (np.arange(H) + sy0) % 2  # 0 or 1 for each row
    col_mod = (np.arange(W) + sx0) % 2  # 0 or 1 for each col
    rm = row_mod[:, None]  # (H, 1)
    cm = col_mod[None, :]  # (1, W)

    # Boolean masks for each channel — shape (H, W)
    m0   = (rm == 1) & (cm == 1)  # 0°
    m45  = (rm == 0) & (cm == 1)  # 45°
    m90  = (rm == 0) & (cm == 0)  # 90°
    m135 = (rm == 1) & (cm == 0)  # 135°

    # Vectorised mean over spatial pixels for each frame
    f = frames.astype(np.float64)  # (N, H, W)
    c0   = f[:, m0].mean(axis=1)
    c45  = f[:, m45].mean(axis=1)
    c90  = f[:, m90].mean(axis=1)
    c135 = f[:, m135].mean(axis=1)

    return c0, c45, c90, c135

## Select and load data

Uncomment **one** of the three loaders below.

In [ ]:
# ── Pick ONE loader ────────────────────────────────────────────────────────────

# (c0, c45, c90, c135), t = load_recorder_raw(Path("..."))
# (c0, c45, c90, c135), t = load_cycler_raw(Path("..."))
# (c0, c45, c90, c135), t = load_cycler_means(Path("..."))

# print(f"Loaded {len(t)} frames, duration {t[-1]:.2f} s")

## Channel correction (a-factors)

Enforce $I_0 + I_{90} = I_{45} + I_{135}$ by finding scale factors $a_{90}, a_{45}, a_{135}$ (with $a_0 = 1$).

No T-matrix applied.

In [ ]:
from scipy.optimize import minimize


def channel_correction(c0, c45, c90, c135, b=(0, 0, 0, 0), dec=1000):
    """Apply background subtraction + a-factor correction.

    b = (b_0, b_45, b_90, b_135) — manual background offsets.
    dec — decimation factor for fitting (use every dec-th sample).

    Returns corrected (c0, c45, c90, c135).
    """
    # Background subtraction
    c0   = c0   - b[0]
    c45  = c45  - b[1]
    c90  = c90  - b[2]
    c135 = c135 - b[3]

    # Fit a-factors on decimated data
    sl = slice(None, None, max(1, dec))
    ac = np.vstack((c0[sl], c90[sl], c45[sl], c135[sl]))  # (4, M)

    def cost(params, ac):
        a90, a45, a135 = params
        s0   = ac[0]
        s90  = ac[1] * a90
        s45  = ac[2] * a45
        s135 = ac[3] * a135
        return np.sum((s0 + s90 - s45 - s135) ** 2)

    res = minimize(cost, [1.0, 1.0, 1.0], args=(ac,), method="Nelder-Mead")
    a90, a45, a135 = res.x
    print(f"a-factors: a90={a90:.5f}  a45={a45:.5f}  a135={a135:.5f}  (residual={res.fun:.2e})")

    # Apply
    c90  = c90  * a90
    c45  = c45  * a45
    c135 = c135 * a135

    return c0, c45, c90, c135

In [ ]:
c0, c45, c90, c135 = channel_correction(
    c0, c45, c90, c135,
    b=(b_0, b_45, b_90, b_135),
)

## Anisotropy + centering → φ, θ

In [ ]:
# ── Fourkas model ─────────────────────────────────────────────────────────────
def fourkas_ABC(alpha):
    """Fourkas (2001) A, B, C as a function of collection half-angle α."""
    ca = np.cos(alpha)
    A = 1/6  - ca/4        + ca**3/12
    B =        ca/8        - ca**3/8
    C = 7/48 - ca/16 - ca**2/16 - ca**3/48
    return A, B, C


def anisotropy(c0, c45, c90, c135, center=(0.0, 0.0)):
    """Compute anisotropy components, r, φ (wrapped), and φ (unwrapped)."""
    denom_x = c0 + c90
    denom_y = c45 + c135
    ax = np.where(denom_x > 0, (c0 - c90)   / denom_x, 0.0) - center[0]
    ay = np.where(denom_y > 0, (c45 - c135) / denom_y, 0.0) - center[1]

    r = np.sqrt(ax**2 + ay**2)
    phi_wrapped  = 0.5 * np.arctan2(ay, ax) % np.pi   # [0, π)  — π-degenerate
    phi_unwrapped = np.unwrap(0.5 * np.arctan2(ay, ax), period=np.pi)

    return ax, ay, r, phi_wrapped, phi_unwrapped


def theta_from_r(r, alpha):
    """Invert Fourkas model: r → θ.  Returns θ in radians [0, π/2]."""
    A, B, C = fourkas_ABC(alpha)
    r_sat = C / (A + B)
    r_clip = np.clip(r, 0.0, r_sat)
    denom = C - r_clip * B
    denom_safe = np.where(denom > 0, denom, np.nan)
    sinsq = np.clip((r_clip * A) / denom_safe, 0.0, 1.0)
    return np.arcsin(np.sqrt(sinsq))

In [ ]:
# ── Compute ───────────────────────────────────────────────────────────────────
alpha = np.arcsin(NA_obj / nw)
A_cal, B_cal, C_cal = fourkas_ABC(alpha)
r_sat = C_cal / (A_cal + B_cal)
print(f"NA={NA_obj}, n={nw} → α={np.degrees(alpha):.2f}°, r_sat={r_sat:.5f}")

ax, ay, r, phi_w, phi_u = anisotropy(c0, c45, c90, c135, center=(center_ax, center_ay))
theta = theta_from_r(r, alpha)

print(f"r:     mean={np.nanmean(r):.4f}  max={np.nanmax(r):.4f}")
print(f"φ:     mean={np.degrees(np.nanmean(phi_w)):.1f}°")
print(f"θ:     mean={np.degrees(np.nanmean(theta)):.1f}°  max={np.degrees(np.nanmax(theta)):.1f}°")

## Plots

In [ ]:
dec = max(1, len(t) // 5000)  # decimate for plotting

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

axes[0].plot(t[::dec], np.degrees(phi_u[::dec]), ",", alpha=0.5)
axes[0].set_ylabel("φ (°, unwrapped)")
axes[0].set_title("Azimuthal angle φ vs time")

axes[1].plot(t[::dec], np.degrees(theta[::dec]), ",", alpha=0.5)
axes[1].set_ylabel("θ (°)")
axes[1].set_xlabel("Time (s)")
axes[1].set_title("Polar angle θ vs time")
axes[1].set_ylim(0, 90)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# φ histogram — wrapped to [0, 2π) via 2*phi_wrapped
phi_2pi = (2 * phi_w) % (2 * np.pi)  # map π-degenerate → full 2π
axes[0].hist(np.degrees(phi_2pi), bins=180, range=(0, 360), density=True, alpha=0.7)
axes[0].set_xlabel("2φ (°)")
axes[0].set_ylabel("Density")
axes[0].set_title("φ histogram (wrapped to 2π)")
axes[0].set_xlim(0, 360)

# θ histogram — [0, π/2]
theta_valid = theta[~np.isnan(theta)]
axes[1].hist(np.degrees(theta_valid), bins=90, range=(0, 90), density=True, alpha=0.7)
axes[1].set_xlabel("θ (°)")
axes[1].set_ylabel("Density")
axes[1].set_title("θ histogram")
axes[1].set_xlim(0, 90)

plt.tight_layout()
plt.show()